In [6]:
!pip install -q -U gradio deep-translator gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.6 MB/s eta 0:00:00


In [13]:
import gradio as gr
from deep_translator import GoogleTranslator
from gtts import gTTS
import tempfile

# Commonly used languages
LANGUAGES = {
    "Auto Detect": "auto",
    "English": "en",
    "Hindi": "hi",
    "Spanish": "es",
    "French": "fr",
    "German": "de",
    "Chinese": "zh-CN",
    "Japanese": "ja",
    "Punjabi": "pa",
    "Bengali": "bn",
    "Arabic": "ar",
    "Portuguese": "pt",
    "Russian": "ru",
    "Italian": "it",
    "Urdu": "ur",
}

# Translate text
def translate_text(text, source, target):
    if not text or not text.strip():
        return "Please enter text to translate."

    try:
        if source == target:
            return text

        translator = GoogleTranslator(
            source=source,
            target=target
        )
        result = translator.translate(text)
        return result or "No translation returned."

    except Exception as e:
        return f"Translation failed: {str(e)}"


# Generate speech
def speak_text(text, language):
    if not text or not text.strip():
        raise gr.Error("Please translate some text first.")

    if language == "auto":
        language = "en"

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".mp3", delete=False
        ) as audio_file:
            filename = audio_file.name

        gTTS(text=text, lang=language).save(filename)
        return filename

    except Exception as e:
        raise gr.Error(f"Audio error: {str(e)}")


# Dropdowns
source_options = list(LANGUAGES.keys())
target_options = [
    name for name in LANGUAGES if name != "Auto Detect"
]

with gr.Blocks() as demo:
    gr.Markdown("# 🌍 Language Translation Tool")
    gr.Markdown("Translate text between different languages.")

    with gr.Row():
        source = gr.Dropdown(
            choices=source_options,
            value="Auto Detect",
            label="Source Language"
        )
        target = gr.Dropdown(
            choices=target_options,
            value="Hindi",
            label="Target Language"
        )

    input_box = gr.Textbox(
        label="Enter Text",
        placeholder="Type your text here...",
        lines=4
    )

    translate_btn = gr.Button("Translate", variant="primary")
    output_box = gr.Textbox(
    label="Translated Text",
    lines=4,
    interactive=False
)

    translate_btn.click(
        fn=lambda text, src, tgt: translate_text(
            text,
            LANGUAGES[src],
            LANGUAGES[tgt]
        ),
        inputs=[input_box, source, target],
        outputs=output_box
    )

    listen_btn = gr.Button("🔊 Listen to Translation")
    audio_box = gr.Audio(label="Speech", type="filepath")

    listen_btn.click(
        fn=lambda text, tgt: speak_text(
            text,
            LANGUAGES[tgt]
        ),
        inputs=[output_box, target],
        outputs=audio_box
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9fe3cd69e1ec9a029.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
